# BE-HPG — Phase 6: Revision (bootstrap CIs + SSP ablation + failure figure)

Round-2 reviewer items handled here, in **one run**:
1. **Bootstrap 95% confidence intervals** over the fixed test episodes (re-evaluates the
   already-trained checkpoints with LCC, logging per-slice metrics — no retraining).
2. **SSP ablation** — trains **BE-HPG without the SSP hard prototypes** (`configs/be_hpg_nossp.yaml`,
   `use_ssp=false`), same backbone/loss/episodes, evaluated with the same LCC.
3. **Pre-LCC failure figure** — renders BE-HPG slices with off-target activations that LCC removes.

**Run:** GPU runtime. Set `SMOKE=True` first to verify end-to-end (~2 min), then `SMOKE=False`
for the real run (eval ~10 min + one ~2–4 min training, resumable). Upload `be-hpg-src.zip` at
cell 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/be-hpg'
print('Drive:', DRIVE_ROOT)

In [ ]:
!pip -q install timm medpy scikit-learn

In [ ]:
import os, sys, shutil
FORCE_UPLOAD = True
zip_drive = f'{DRIVE_ROOT}/be-hpg-src.zip'
if FORCE_UPLOAD or not os.path.exists(zip_drive):
    from google.colab import files
    print('Upload be-hpg-src.zip:')
    up = files.upload()
    with open(zip_drive, 'wb') as f:
        f.write(up[list(up)[0]])
code_dir = '/content/be-hpg'
shutil.rmtree(code_dir, ignore_errors=True); os.makedirs(code_dir, exist_ok=True)
os.system(f'unzip -q -o "{zip_drive}" -d "{code_dir}"')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]        # drop cached modules so a re-uploaded zip takes effect
import inspect
from src.models.be_hpg import BEHPG
from src.utils.bootstrap import bootstrap_ci
ok = ('use_ssp' in inspect.signature(BEHPG.__init__).parameters) and callable(bootstrap_ci)
print('code ready | revision build loaded =', ok)

In [ ]:
import torch, json, time
import numpy as np
import pandas as pd
from src.utils.config import load_config
from src.utils.seed import set_seed
from src.engine.build import build_model
from src.engine.eval import eval_episodes
from src.engine.train import train_episodes
from src.engine.checkpoint import load_checkpoint
from src.data.sampler import sampler_from_npz
from src.utils.bootstrap import bootstrap_ci
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

SMOKE = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
suffix = '_smoke' if SMOKE else ''
NPZ = f'{DRIVE_ROOT}/data/spleen/spleen{suffix}.npz'
MANIFEST = f'{DRIVE_ROOT}/data/spleen/manifest{suffix}.csv'
RES = f'{DRIVE_ROOT}/results'; os.makedirs(RES, exist_ok=True)
mdf = pd.read_csv(MANIFEST) if os.path.exists(MANIFEST) else None
TEST_SPLIT  = None if SMOKE else 'test'
TRAIN_SPLIT = None if SMOKE else 'train'
EVAL_EP = 10 if SMOKE else 150
EPISODE_SEED = 1234
print('device:', DEVICE, '| SMOKE', SMOKE, '| eval episodes', EVAL_EP)

def eval_with_samples(model, cfg):
    """Raw + LCC summaries, plus per-slice metric lists (LCC) for bootstrap CIs."""
    out = {'by_shot': {}, 'by_shot_lcc': {}}; samples = {}
    qs = cfg['episode']['query_size']
    for k in (1, 5):
        ev = sampler_from_npz(NPZ, mdf, TEST_SPLIT, k_shots=[k], query_size=qs, seed=EPISODE_SEED)
        out['by_shot'][f'{k}shot'] = eval_episodes(model, ev, episodes=EVAL_EP, k_shot=k, device=DEVICE)
        ev = sampler_from_npz(NPZ, mdf, TEST_SPLIT, k_shots=[k], query_size=qs, seed=EPISODE_SEED)
        summ, raw = eval_episodes(model, ev, episodes=EVAL_EP, k_shot=k, device=DEVICE,
                                  postprocess='lcc', return_samples=True)
        out['by_shot_lcc'][f'{k}shot'] = summ; samples[f'{k}shot'] = raw
    return out, samples

def ci_line(name, samp):
    d, lo_d, hi_d = bootstrap_ci(samp['1shot']['dice'])
    h, lo_h, hi_h = bootstrap_ci(samp['1shot']['hd95'])
    print('%-16s 1-shot Dice %.3f [%.3f, %.3f] | HD95 %.1f [%.1f, %.1f]'
          % (name, d, lo_d, hi_d, h, lo_h, hi_h))

## Part A — Bootstrap 95% confidence intervals (no retraining)
Re-evaluates the three trained checkpoints on the fixed test episodes with LCC, logging per-slice
metrics, and reports percentile-bootstrap CIs. Saves `<model>_spleen_lcc.json` (summary) and
`<model>_spleen_lcc_samples.json` (per-slice lists).

In [ ]:
MODELS = {'protonet': 'protonet.yaml', 'meta_reweight': 'meta_reweight.yaml', 'be_hpg': 'be_hpg.yaml'}
print('==== 95% bootstrap CIs (1-shot, LCC) ====')
for name, yml in MODELS.items():
    cfg = load_config(f'{code_dir}/configs/{yml}')
    model = build_model(cfg)
    ckpt = f'{DRIVE_ROOT}/checkpoints/{name}_spleen{suffix}.pt'
    assert os.path.exists(ckpt), f'missing checkpoint {ckpt} (run Phase 4 full first)'
    load_checkpoint(ckpt, model, map_location=DEVICE)
    out, samples = eval_with_samples(model, cfg)
    out['model'] = name
    json.dump(out,     open(f'{RES}/{name}_spleen_lcc{suffix}.json', 'w'), indent=2)
    json.dump(samples, open(f'{RES}/{name}_spleen_lcc_samples{suffix}.json', 'w'), indent=2)
    ci_line(name, samples)
print('saved *_spleen_lcc.json and *_spleen_lcc_samples.json')

## Part B — SSP ablation: BE-HPG without the hard prototypes
Trains `configs/be_hpg_nossp.yaml` (`use_ssp=false`) — identical backbone, boundary loss,
class-balanced BCE, and episode budget — then evaluates with the same LCC. Resumable.

In [ ]:
cfg = load_config(f'{code_dir}/configs/be_hpg_nossp.yaml')
assert cfg['model']['ssp']['use_ssp'] is False, 'be_hpg_nossp must have use_ssp=false'
set_seed(cfg.get('seed', 42))
TRAIN_EP = 40 if SMOKE else cfg['train']['episodes']      # 1500, matches the full model
tag = 'be_hpg_nossp'
model = build_model(cfg)
tr = sampler_from_npz(NPZ, mdf, TRAIN_SPLIT, k_shots=cfg['episode']['k_shot_train'],
                      query_size=cfg['episode']['query_size'], seed=cfg.get('seed', 42))
opt = AdamW(model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
sched = CosineAnnealingLR(opt, T_max=TRAIN_EP)
ckpt = f'{DRIVE_ROOT}/checkpoints/{tag}_spleen{suffix}.pt'
start = 0
if (not SMOKE) and os.path.exists(ckpt):
    start, _ = load_checkpoint(ckpt, model, opt, sched, map_location=DEVICE)
t0 = time.time()
if start < TRAIN_EP:
    train_episodes(model, tr, episodes=TRAIN_EP, start_episode=start, optimizer=opt, scheduler=sched,
                   lam=cfg['loss']['boundary_lambda'], device=DEVICE, amp=cfg.get('amp', True),
                   ckpt_path=ckpt, ckpt_every=250, log_every=max(TRAIN_EP // 5, 1),
                   balanced=cfg['loss'].get('balanced_bce', True),
                   max_pw=cfg['loss'].get('bce_max_pos_weight', 20.0))
print(f'trained no-SSP in {time.time() - t0:.0f}s')
out, samples = eval_with_samples(model, cfg); out['model'] = tag
json.dump(out,     open(f'{RES}/{tag}_spleen_lcc{suffix}.json', 'w'), indent=2)
json.dump(samples, open(f'{RES}/{tag}_spleen_lcc_samples{suffix}.json', 'w'), indent=2)

full = json.load(open(f'{RES}/be_hpg_spleen_lcc{suffix}.json'))['by_shot_lcc']['1shot']
nossp = out['by_shot_lcc']['1shot']
print('\n==== SSP ablation (1-shot, LCC) ====')
print('BE-HPG (full, +SSP) : Dice %.3f | IoU %.3f | HD95 %.1f' % (full['dice'], full['iou'], full['hd95']))
print('BE-HPG w/o SSP      : Dice %.3f | IoU %.3f | HD95 %.1f' % (nossp['dice'], nossp['iou'], nossp['hd95']))
ci_line('be_hpg (full)', json.load(open(f'{RES}/be_hpg_spleen_lcc_samples{suffix}.json')))
ci_line('be_hpg_nossp', samples)

## Part C — Pre-LCC failure figure (off-target activations)
Finds held-out query slices where the **raw** BE-HPG mask has off-target components that LCC
removes, and renders `query | ground truth | raw | after-LCC`. Download `prelcc_failure.png`
into `paper/figures/`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from src.metrics.postprocess import largest_connected_component

cfg = load_config(f'{code_dir}/configs/be_hpg.yaml')
model = build_model(cfg)
load_checkpoint(f'{DRIVE_ROOT}/checkpoints/be_hpg_spleen{suffix}.pt', model, map_location=DEVICE)
model.to(DEVICE).eval()
ev = sampler_from_npz(NPZ, mdf, TEST_SPLIT, k_shots=[1], query_size=cfg['episode']['query_size'], seed=EPISODE_SEED)

cands = []
with torch.no_grad():
    for _ in range(EVAL_EP):
        b = ev.sample(k=1)
        si, sm, qi, qm = (b['sup_img'].to(DEVICE), b['sup_mask'].to(DEVICE),
                          b['qry_img'].to(DEVICE), b['qry_mask'].to(DEVICE))
        prob = torch.sigmoid(model(si, sm, qi))
        for q in range(prob.shape[0]):
            raw = (prob[q, 0] > 0.5).float().cpu().numpy()
            gt = qm[q, 0].cpu().numpy()
            lcc = largest_connected_component(raw)
            _, ncomp = ndimage.label(raw)
            offtarget = float(raw.sum() - lcc.sum())
            if ncomp >= 2 and lcc.sum() > 0 and offtarget > 0 and gt.sum() > 0:
                cands.append((offtarget, qi[q, 0].cpu().numpy(), gt, raw, lcc))
cands.sort(key=lambda c: -c[0])
pick = cands[:2]
print('off-target candidates found:', len(cands), '| showing', len(pick))

if pick:
    fig, axes = plt.subplots(len(pick), 4, figsize=(8, 2.3 * len(pick)))
    if len(pick) == 1:
        axes = axes[None, :]
    titles = ['Query', 'Ground truth', 'Raw BE-HPG', 'After LCC']
    for r, (sc, img, gt, raw, lcc) in enumerate(pick):
        off = (raw > 0.5) & (lcc < 0.5)
        for c, t in enumerate(titles):
            ax = axes[r, c]; ax.imshow(img, cmap='gray'); ax.axis('off')
            if r == 0:
                ax.set_title(t, fontsize=11)
            if c == 1:
                ax.contour(gt, colors='lime', linewidths=1.0)
            if c == 2:
                ax.contour((raw > 0.5).astype(float), colors='red', linewidths=1.0)
                ys, xs = np.where(off)
                if len(xs):
                    ax.annotate('off-target', xy=(xs.mean(), ys.mean()),
                                xytext=(xs.mean() + 30, ys.mean() + 30), color='yellow', fontsize=8,
                                arrowprops=dict(color='yellow', arrowstyle='->', lw=1.2))
            if c == 3:
                ax.contour((lcc > 0.5).astype(float), colors='red', linewidths=1.0)
    fig.tight_layout()
    fig.savefig(f'{RES}/prelcc_failure.png', dpi=140, bbox_inches='tight')
    print('saved', f'{RES}/prelcc_failure.png')
else:
    print('No off-target cases at this budget; try SMOKE=False (raw HD95 ~44 px implies they exist).')

## Report back
Download from Drive `results/` into the repo `results/` folder:
- `protonet_spleen_lcc_samples.json`, `meta_reweight_spleen_lcc_samples.json`,
  `be_hpg_spleen_lcc_samples.json`
- `be_hpg_nossp_spleen_lcc.json` and `be_hpg_nossp_spleen_lcc_samples.json`
- `prelcc_failure.png`  →  put this one in `paper/figures/`

Then paste the printed **CI table**, the **SSP ablation** lines, and the candidate count.
I'll regenerate `paper/tab_ci.tex` / `paper/tab_ssp.tex` (`python -m src.utils.aggregate`) and
finalise the paper.